In [ ]:
from __future__ import annotations

from pathlib import Path
import sys, os
from dataclasses import dataclass, field

import torch
import torch.nn as nn
from torch import Tensor
import torch.nn.functional as F
from accelerate import Accelerator
from accelerate.utils import ProjectConfiguration

from src.core.config import Config, TrainConfig, LoggingConfig, OptimConfig, WandBConfig
from src.core import build_dataloaders, build_optim, build_lr_scheduler, build_loss_fn, build_callbacks
from src.core.registry import get_dataset
from src.tasks.sort_of_clevr.spec import SortOfClevrDataConfig
from src.tasks.sqoop.spec import SqoopDataConfig
from src.training import Trainer
from src.utils import set_seed, set_torch_config

from src.models.syncnet import SyncNet, SyncNetConfig, ABLATIONS


from src.core.contracts import VQABatch, VQAOutput


In [ ]:
def train(
        model_class, 
        model_cfg,
        cfg: Config, 
        out_dir: str
    ):

    model = model_class.from_config(
        model_cfg, cfg.dataset.name, int(get_dataset(cfg).ANSWER_DIM)
        )
    
    os.makedirs(out_dir, exist_ok=True)
    set_seed(cfg.train.seed)
    accelerator = Accelerator(
        mixed_precision=cfg.train.mixed_precision,
        gradient_accumulation_steps=cfg.train.grad_accum,
        project_config=ProjectConfiguration(project_dir=out_dir),
    )

    set_torch_config(str(accelerator.device))
    dataloaders = build_dataloaders(cfg, str(accelerator.device))
    optimiser = build_optim(model, cfg.optim)
    scheduler = build_lr_scheduler(optimiser, cfg.train.n_steps, cfg.optim)

    try:
        trainer = Trainer(
            cfg=cfg, 
            out_dir=out_dir, 
            logger=None, 
            model=model, 
            dataloaders=dataloaders,
            optimiser=optimiser, 
            scheduler=scheduler, 
            accelerator=accelerator,
            callbacks=build_callbacks(cfg, model), 
            loss_fn=build_loss_fn(cfg),
        )
        return trainer.train(), trainer
    
    finally:
        accelerator.end_training()


In [ ]:
OUT_DIR = Path.cwd() / 'notebooks' / 'outputs'
DATA = str(Path.cwd() / 'data')

train_cfg = TrainConfig(
    seed=0,
    n_steps=100_000,               
    train_bs=256, 
    val_bs=1024,
    early_stop_metric='loss', 
    early_stop_big_is_better=False,
    early_stop_patience=None, 
    early_stop_min_delta=0.0,
    mixed_precision='bf16', 
    compile_model=True,
    grad_accum=1, 
    grad_clip=1.0, 
    loader_mode='gpu_cached', 
    num_workers=0,
)
logging_cfg = LoggingConfig(
    eval_log_interval=500, 
    train_log_interval=100, 
    info_metrics=['loss', 'accuracy'], 
    save_best=False
    )
optim_cfg = OptimConfig(
    optimiser='adamw', 
    lr=3e-4, 
    weight_decay=0.01, 
    lr_scheduler='warmup_cosine', 
    lr_scheduler_params={'warmup_steps': 300}
    )
wandb_cfg = WandBConfig(enabled=False)
callbacks = ['freeze_phase', 'zero_phase', 'shuffle_phase', {'name': 'test_time_t_override', 't_values': [2, 8], 'n_repeats': 1, 'max_batches': 4}]   # accuracy is always on


CFG = {
    'soc': Config(
        train=train_cfg,
        logging = LoggingConfig(
            eval_log_interval=500, 
            train_log_interval=100, 
            info_metrics=[
                'loss', 
                'accuracy', 
                'non_relational_accuracy', 
                'binary_accuracy', 
                'ternary_accuracy'
                ], 
            save_best=False
            ),
        wandb=wandb_cfg,
        optim=optim_cfg,
        dataset=SortOfClevrDataConfig(
            seed=1, 
            root=DATA, 
            train_size=36000, 
            test_size=1_000, 
            nb_questions=10, 
            t_subtype=-1
            ),
        callbacks=callbacks
        ),
    'sqoop_rhs1': Config(
        train=train_cfg,
        logging=logging_cfg,
        wandb=wandb_cfg,
        optim=optim_cfg,
        dataset=SqoopDataConfig(
            seed=0, 
            root=DATA, 
            train_size=1_080_000, 
            test_size=25_600, 
            rhs_variety=1
            ),
        callbacks=callbacks
        ),
}


In [ ]:
class CNNEncoderSmall(nn.Module):

    def __init__(self, img_size: int, ch: int = 24) -> None:
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(3, ch, 3, stride=2, padding=1),
            nn.BatchNorm2d(ch),
            nn.ReLU(),

            nn.Conv2d(ch, ch, 3, stride=2, padding=1),
            nn.BatchNorm2d(ch),
            nn.ReLU(),
        )

        spatial = img_size
        for _ in range(2):
            spatial = (spatial + 1) // 2

        self.spatial = spatial
        self.n_tokens = spatial * spatial
        self.ch = ch

    def forward(self, x: Tensor) -> Tensor:
        # -> (B, self.ch, self.spatial, self.spatial)
        return self.cnn(x)


class SyncNetTrunk(SyncNet):

    def build_stem(self, cfg):
        return CNNEncoderSmall({'sort_of_clevr': 75, 'sqoop': 64}[self.dataset], cfg.field_ch)

In [ ]:
MODEL_CFG = {
    'soc': SyncNetConfig(encoder={'name': 'cnn', 'ch': 128}, bias_init='zero'),
    'sqoop_rhs1': SyncNetConfig(encoder={'name': 'cnn', 'ch': 64}, bias_init='partition'),
}
MODEL = SyncNet         


In [ ]:
results = {}
for task, cfg in CFG.items():
    results[task], trainer = train(MODEL, MODEL_CFG[task], cfg, str(OUT_DIR / task))


In [ ]:
for task, res in results.items():
    print(task, res)